In [11]:
import pandas as pd

In [12]:
def get_clust_dfs(parquet_file):
    
    df = pd.read_parquet(parquet_file)
    
    clust_df = pd.DataFrame(df.groupby('clust')['time'].value_counts()).rename(columns={'time': 'count'})
    clust_df = clust_df.reset_index().pivot(index='clust', columns='time', values='count')
    clust_df['total'] = clust_df['t1'] + clust_df['t2']
    clust_df['prop_t2'] = clust_df['t2'] / clust_df['total']

    return (df, clust_df)

pd.set_option('display.max_colwidth', None)

### Explore cluster examples for one word

This code is intended to be used interactively: load the parquet file corresponding to a word of interest, and then manually inspect sentences from different clusters, typically focusing on the clusters with the strongest difference in temporal distribution.

In [13]:
df, clust_df = get_clust_dfs('cluster_out/ensure_VERB.parquet')

In [14]:
# Check distribution across clusters
clust_df.sort_values('prop_t2', ascending=False).round(2)

time,t1,t2,total,prop_t2
clust,,,,
3,94,210,304,0.69
4,156,207,363,0.57
2,62,77,139,0.55
6,126,142,268,0.53
1,116,104,220,0.47
5,71,58,129,0.45
7,278,153,431,0.35
0,97,49,146,0.34


In [15]:
# Check individual examples
df[df['clust'] == 4][:10]

,sent_id,sent,clust,dist,time
809,440808,"To ensure the high quality of the annotations , we had each sentence annotated at least four times by different users in each round .",4,0.023664,t2
810,285492,"Finally , to ensure the quality of all annotations , we cross - matched the labels of the common domains by consulting both Snopes and Media Bias / Fact Check .",4,0.026875,t1
811,289576,"( 12 ) Inspired by Zhao et al . ( 2017 ) , to ensure the accuracy of the selected entity , we enforce the relevance between the selected entity and the ground truth summary .",4,0.028257,t1
812,515138,"Second , we revised those test cases both in automatic or manual manner to ensure the quality of the test suite , based on the following protocols .",4,0.028593,t2
813,48484,"To ensure the high quality of the annotation procedure , we manually annotated a set of 200 control tasks .",4,0.029419,t1
814,495679,"To ensure the correct balance of LLM utility and safety , we created three small test sets : 1 ) Task - Harmful .",4,0.030667,t2
815,345093,"To ensure the fairness of the experiment , we concatenated the question type , question topic , and answer to form the input of the baseline model .",4,0.030957,t2
816,150805,"To ensure the high quality of the dataset , we employ five real insurance assessors to fill in the insurance report by selecting a string fragment of the utterance from the given multi - turn dialogue .",4,0.031477,t1
817,485140,"To ensure the reliability of few - shot baselines , the results of ProtoBERT and StructShot are the average results of 100 runs .",4,0.032020,t2
818,352830,We do this to ensure fairness in the comparison since the number of activation - based style vectors is significantly higher than the number of training - based vectors .,4,0.032514,t2


### Prepare examples for paper

In [16]:
examples = {
    'ensure_VERB': [
        [156946, 292912, 425267, 84889], # so as
        [48484, 157291, 302558, 317917] # to guarantee
    ],
    'utilize_VERB': [
        [251359, 251492, 118078, 321579], # use to the full potential
        [69827, 16261, 150784, 212900, 105923] # use
    ],
    'crucial_ADJ': [
        [327523, 67753, 292406, 281655], # very impotant in determining the outcome of sth
        [1124136, 832670, 881967, 771890, 495857] # very important in general
    ],
    'notably_ADV': [
        [2368709, 177374, 153629, 935066], # especially, particularly; of note
        [1272006, 1510685, 1761158, 1837590, 2192638] # in a notable manner, to a high degree (very/much: intensifier)
    ]
}

In [17]:
latex_outs = []

for target in examples:
    latex_out = []
    stats_pretty = ''
    df, clust_df = get_clust_dfs(f'cluster_out/{target}.parquet')
    top = True
    
    for cluster_sents in examples[target]:
        
        # Get cluster ID based on sentence IDs
        clust_id = df[df['sent_id'].isin(cluster_sents)]['clust'].unique()
        assert len(clust_id) == 1
        clust_id = clust_id[0]
        
        # Get cluster statistics
        total, prop_t2 = clust_df.iloc[clust_id][['total', 'prop_t2']]
        total = round(total)
        perc_t1 = round((1-prop_t2)*100)
        perc_t2 = round(prop_t2*100)
        stats_comment = f'\n% {total} sents -- {perc_t1}% t1 / {perc_t2}% t2 -- clust {clust_id}'
        if top:
            clust_str = 'Top'
            top = False
        else:
            clust_str = 'Bottom'
        stats_pretty += rf'{clust_str}: {perc_t1}\%\,\tone vs.\ {perc_t2}\%\,\ttwo ({total} sentences). '
        
        latex_out.append(stats_comment)

        # Get sentences
        win = 8
        sent_target = target.split('_')[0]
        sents = df[df['sent_id'].isin(cluster_sents)]['sent'].to_list()
        out_sents = []
        for sent in sents:
            out_sent = []
            tmp = []
            for tok in sent.split():
                if tok.lower() == sent_target:
                    out_sent.append(' '.join(tmp[-win:]))
                    # out_sent.append(tok)
                    out_sent.append('\\textbf{' + tok + '}')
                    tmp = []
                else:
                    tmp.append(tok)
            out_sent.append(' '.join(tmp[:win]))
            out_sents.append(' & '.join(out_sent) + ' \\\\')
        latex_out += out_sents
    latex_outs += latex_out
    latex_outs.append('\n'+stats_pretty)

In [18]:
for l in latex_outs:
    print(l)


% 431 sents -- 65% t1 / 35% t2 -- clust 7
We add an adversarial objective to & \textbf{ensure} & that the model focuses on language - agnostic \\
measure the semantic similarity between generated questions to & \textbf{ensure} & that the questions assess the same content . \\
sampling process , we store this data to & \textbf{ensure} & that every model processes exactly the same set \\
we diversify the output of each task to & \textbf{ensure} & that the model can provide a variety of \\

% 363 sents -- 43% t1 / 57% t2 -- clust 4
To & \textbf{ensure} & the high quality of the annotation procedure , \\
To & \textbf{ensure} & the quality of data , they also calculated \\
Finally , to & \textbf{ensure} & training stability and prevent overfitting , we modify \\
the hyperparameters the same for different models to & \textbf{ensure} & the fairness of the experiment . \\

Top: 65\%\,\tone vs.\ 35\%\,\ttwo (431 sentences). Bottom: 43\%\,\tone vs.\ 57\%\,\ttwo (363 sentences). 

% 281 sents --